|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>Detokenization<h1>|
|<h2>Lecture:</h2>|<h1><b>Why you cannot decode tokens one at a time<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B')
print('vocab', tok.vocab_size)

/home/venugopalan/vllm-from-scratch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


vocab 151643


# You cannot decode tokens one at a time

Streaming means emitting text as each token arrives. The obvious
implementation is `decode(token)` per token, concatenated.

It works for most text, which is why it ships.

In [2]:
def naive_stream(ids):
  return ''.join(tok.decode([i]) for i in ids)

for text in ['Hello world, how are you?', 'print("hi")  # done']:
  ids = tok(text, add_special_tokens=False).input_ids
  print(f'{text!r}')
  print(f'  naive == batch: {naive_stream(ids) == tok.decode(ids)}')

'Hello world, how are you?'
  naive == batch: True
'print("hi")  # done'
  naive == batch: True


### And then it does not

In [3]:
broken = ['\U0001F468\u200d\U0001F469\u200d\U0001F467\u200d\U0001F466 family',
          '\U0001F3F3\ufe0f\u200d\U0001F308 pride']

for text in broken:
  ids = tok(text, add_special_tokens=False).input_ids
  naive, correct = naive_stream(ids), tok.decode(ids)
  print(f'pieces : {tok.convert_ids_to_tokens(ids)}')
  print(f'naive  : {naive!r}')
  print(f'correct: {correct!r}')
  print(f'match  : {naive == correct}\n')

pieces : ['ðŁĳ¨', 'âĢ', 'į', 'ðŁĳ©', 'âĢ', 'į', 'ðŁĳ§', 'âĢ', 'į', 'ðŁĳ¦', 'Ġfamily']
naive  : '👨��👩��👧��👦 family'
correct: '👨\u200d👩\u200d👧\u200d👦 family'
match  : False

pieces : ['ðŁı³', 'ï¸ı', 'âĢ', 'į', 'ðŁĮĪ', 'Ġpride']
naive  : '🏳️��🌈 pride'
correct: '🏳️\u200d🌈 pride'
match  : False



Look at the pieces. The zero-width joiner that glues the family emoji
together is three bytes, and the tokenizer split it across two tokens. Decode
either of them alone and you get a replacement character, because half a
character is not a character.

The same thing happens to accented Latin, to CJK, and to anything else whose
UTF-8 encoding is longer than one byte. It happens rarely enough to survive
testing and often enough to be reported as mojibake by users.

# Decode the prefix, emit the difference

Keep the token ids. Decode all of them. Emit whatever is longer than what you
emitted last time.

The tokenizer sees whole characters again, because it is looking at whole
sequences.

In [4]:
class Incremental:
  def __init__(self, tokenizer):
    self.tok     = tokenizer
    self.ids     = []
    self.emitted = 0

  def push(self, token_id):
    self.ids.append(token_id)
    text = self.tok.decode(self.ids)
    # a trailing replacement character means the last character is not
    # finished yet. Hold it back; the next token will complete it.
    if text.endswith('\ufffd'):
      return ''
    out = text[self.emitted:]
    self.emitted = len(text)
    return out

  def flush(self):
    """The stream ended. Emit whatever is left, broken or not."""
    text = self.tok.decode(self.ids)
    out, self.emitted = text[self.emitted:], len(text)
    return out

for text in broken + ['caf\u00e9 na\u00efve \u65e5\u672c\u8a9e']:
  ids = tok(text, add_special_tokens=False).input_ids
  d = Incremental(tok)
  streamed = ''.join(d.push(i) for i in ids) + d.flush()
  print(f'{streamed == tok.decode(ids)}  {streamed!r}')

True  '👨\u200d👩\u200d👧\u200d👦 family'
True  '🏳️\u200d🌈 pride'
True  'café naïve 日本語'


### Fuzz it, because examples prove nothing

A streaming detokenizer has one job and it is an equality: whatever the
stream emits, concatenated, must equal what a batch decode would have
produced. Test that on a lot of text rather than on four strings you thought
of.

In [5]:
import random

rng = random.Random(0)
fails = 0
for trial in range(300):
  ids = [rng.randrange(tok.vocab_size) for _ in range(rng.randint(1, 40))]
  d = Incremental(tok)
  streamed = ''.join(d.push(i) for i in ids) + d.flush()
  if streamed != tok.decode(ids):
    fails += 1
    if fails == 1:
      print('first failure:', tok.convert_ids_to_tokens(ids)[:10])
print(f'{fails} failures in 300 random sequences')

0 failures in 300 random sequences


# The other half: stop strings

A request says "stop when you see `END`". The model emits ` EN` and then
`D`. Neither token contains the stop string and the two of them together do.

So you cannot check tokens. You have to check the **text**, over a window
that spans token boundaries, and you have to hold back anything that might
turn out to be the start of one.

In [6]:
STOP = 'END'

def stream_until_stop(ids, stop):
  """Emit text, stop at `stop`, and never emit a prefix of it."""
  d, out, hold = Incremental(tok), [], ''
  for i in ids:
    hold += d.push(i)
    pos = hold.find(stop)
    if pos >= 0:
      out.append(hold[:pos])
      return ''.join(out), True
    # hold back anything that could still become the stop string
    keep = 0
    for k in range(1, min(len(stop), len(hold))):
      if hold.endswith(stop[:k]): keep = k
    if keep:
      out.append(hold[:-keep]); hold = hold[-keep:]
    else:
      out.append(hold); hold = ''
  return ''.join(out) + hold + d.flush(), False

for text in ['Answer: yes. END OF LINE', 'no stop here at all']:
  ids = tok(text, add_special_tokens=False).input_ids
  emitted, stopped = stream_until_stop(ids, STOP)
  print(f'{text!r}')
  print(f'  emitted {emitted!r}, stopped={stopped}\n')

'Answer: yes. END OF LINE'
  emitted 'Answer: yes. ', stopped=True

'no stop here at all'
  emitted 'no stop here at all', stopped=False



The held-back text is why this is fiddly. Emit eagerly and a user sees `EN`
flash up before you realise it was the beginning of the stop string and cut
the stream. Hold back too much and streaming stutters.

None of this is interesting, and it is the source of most user-visible bugs
in real servers, which is a good reason to get it right once and cover it
with a fuzz test.

    ./vc guide 14